# Masked-Diffusion BabyLM — Strict-Small (Evaluation)

A masked-diffusion denoiser is scored like a masked LM (per-token
pseudo-log-likelihood), so we evaluate it with the official pipeline's **`mlm`**
backend. We run:

1. Full zero-shot + GLUE fine-tuning on the **final** model (`main`).
2. Fast zero-shot on **every** `chck_NM` checkpoint.
3. `collate_preds` to build the submission file.

The model must already be public on the Hub (see `upload_pipeline.ipynb`).

In [ ]:
# Cell 1 — Parameters
MODEL_ID = "<your-user>/babylm-2026-strict-small-mdlm-seed42"
BACKEND  = "mlm"            # masked-diffusion is scored as a masked LM
TRACK    = "strict-small"
EVAL_REPO = "https://github.com/<your-user>/diffusion-babylm-eval.git"

In [ ]:
# Cell 2 — Setup: mount Drive, clone repo, install eval deps, HF token
from google.colab import drive; drive.mount("/content/drive")
import os, subprocess
if not os.path.isdir("/content/diffusion-babylm-eval"):
    subprocess.run(["git", "clone", EVAL_REPO, "/content/diffusion-babylm-eval"], check=True)
%cd /content/diffusion-babylm-eval/strict
!pip install -q -r requirements.txt
!python -m scripts.download_evals   # downloads BLiMP/EWoK/COMPS/entity_tracking/GLUE
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Set HF_TOKEN in Colab Secrets.", e)

In [ ]:
# Cell 3 — FULL zero-shot on the final checkpoint (main)
!./eval_zero_shot.sh $MODEL_ID $BACKEND

In [ ]:
# Cell 4 — FAST zero-shot on all chck_NM checkpoints (intermediate checkpoints)
# Edit the script's revision loop if your checkpoints stop before 100M words.
!./eval_zero_shot_fast_all_revisions.sh $MODEL_ID $BACKEND $TRACK

In [ ]:
# Cell 5 — GLUE fine-tuning on the final checkpoint (full eval requirement)
!./eval_finetune.sh $MODEL_ID

In [ ]:
# Cell 6 — (Optional) diffusion-native scorer: ELBO and the layer-duplication
# variant, which the official pipeline cannot express. Writes predictions.json
# in the same layout so it is interchangeable with the mlm backend above.
%cd /content/diffusion-babylm-eval/diffusion
!python scripts/diffusion_eval_backend.py --model_path_or_name $MODEL_ID \
    --task blimp --data_path ../strict/evaluation_data/full_eval/blimp_filtered \
    --scoring elbo --layer_duplication_factor 2 --backend mlm --save_predictions

In [ ]:
# Cell 7 — Build the submission file (predictions scored server-side)
# collate_preds.sh already passes --fast (required for a full Challenge submission).
%cd /content/diffusion-babylm-eval/strict
!bash scripts/collate_preds.sh $MODEL_ID $BACKEND $TRACK